# Excise 1

##### E01: Original: Train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?
##### 中文翻譯： 訓練一個三字詞（trigram）語言模型，也就是以兩個字元作為輸入，來預測第三個字元。你可以自由選擇使用「統計計數」或「神經網路」來實作。請評估它的損失值（Loss）；它的效果有比雙字詞（bigram）模型更好嗎？

In [78]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

In [37]:
# 首先讀取檔案
words=open('./names.txt','r').read().splitlines()

# 建立字碼對應表
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

V = len(itos)

In [55]:
T = torch.zeros((V**2, V),dtype=torch.int32)
for w in words:
  chs = ['.','.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]

    T[ix1*V + ix2, ix3] += 1



In [ ]:
P = (T+1).float()
P /= P.sum(1, keepdim=True)

log_likelihood = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  chs = ['.', '.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    prob = P[ix1 * V + ix2, ix3]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-504653.)
nll=tensor(504653.)
2.2119739055633545


In [86]:
# 單層神經網路

xs, ys = [], []
for w in words:  
    chs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]

        xs.append(ix1 * V + ix2)
        ys.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

# 初始化網路
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V**2, V), generator=g, requires_grad=True)

for k in range(2000):
    # forward pass
    #xenc = F.one_hot(xs, num_classes=V**2).float()
    #logits = xenc @ W
    # 這裡有一個簡單的寫法
    logits = W[xs] # 這裡的 logits 是一個 (num, V) 的矩陣，代表每個樣本對應到每個字元的 logit 值
    counts = logits.exp() 
    probs = counts / counts.sum(1, keepdim=True) # 修正為 keepdim
    
    # 這裡的 loss 同時包含了 NLL 和 L2 正則化
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    
    # backward pass
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

    if k % 100 == 0: # 每 10 次印一次就好，畫面比較乾淨
        print(f"Step {k}: Loss = {loss.item():.4f}")

Step 0: Loss = 3.8028
Step 100: Loss = 2.5180
Step 200: Loss = 2.3916
Step 300: Loss = 2.3387
Step 400: Loss = 2.3092
Step 500: Loss = 2.2905
Step 600: Loss = 2.2775
Step 700: Loss = 2.2681
Step 800: Loss = 2.2608
Step 900: Loss = 2.2551
Step 1000: Loss = 2.2505
Step 1100: Loss = 2.2468
Step 1200: Loss = 2.2436
Step 1300: Loss = 2.2409
Step 1400: Loss = 2.2386
Step 1500: Loss = 2.2366
Step 1600: Loss = 2.2348
Step 1700: Loss = 2.2332
Step 1800: Loss = 2.2319
Step 1900: Loss = 2.2306


##### E02: Original: Split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?
##### 中文翻譯： 將資料集隨機拆分為 80% 訓練集（train set）、10% 驗證集（dev set）與 10% 測試集（test set）。只在訓練集上訓練 bigram 與 trigram 模型，並分別在驗證集和測試集上評估它們的表現。你觀察到了什麼現象？

In [96]:
def build_dataset(words):
    xs = []
    ys = []

    for w in words:  
        chs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            ix3 = stoi[ch3]

            xs.append(ix1 * V + ix2)
            ys.append(ix3)

    return torch.tensor(xs), torch.tensor(ys), torch.tensor(xs).nelement()

In [ ]:
import random

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
train_words, dev_words, test_words = words[:n1], words[n1:n2], words[n2:]

xs, ys, num = build_dataset(train_words)

xdev, ydev, dev_num = build_dataset(dev_words)

xtest, ytest, test_num = build_dataset(test_words)


# 初始化網路
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V**2, V), generator=g, requires_grad=True)

for k in range(2000):
    # forward pass
    # 這裡有一個簡單的寫法
    logits = W[xs] # 這裡的 logits 是一個 (num, V) 的矩陣，代表每個樣本對應到每個字元的 logit 值
    counts = logits.exp() 
    probs = counts / counts.sum(1, keepdim=True) # 修正為 keepdim
    
    # 這裡的 loss 同時包含了 NLL 和 L2 正則化
    loss = -probs[torch.arange(num), ys].log().mean() #+ 0.01 * (W**2).mean()
    
    # backward pass
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

    if k % 200 == 0: # 每 10 次印一次就好，畫面比較乾淨
        print(f"Step {k}: Loss = {loss.item():.4f}")


Step 0: Loss = 3.7925
Step 100: Loss = 2.5070
Step 200: Loss = 2.3795
Step 300: Loss = 2.3256
Step 400: Loss = 2.2953
Step 500: Loss = 2.2759
Step 600: Loss = 2.2623
Step 700: Loss = 2.2523
Step 800: Loss = 2.2446
Step 900: Loss = 2.2385
Step 1000: Loss = 2.2335
Step 1100: Loss = 2.2294
Step 1200: Loss = 2.2259
Step 1300: Loss = 2.2229
Step 1400: Loss = 2.2203
Step 1500: Loss = 2.2180
Step 1600: Loss = 2.2160
Step 1700: Loss = 2.2142
Step 1800: Loss = 2.2126
Step 1900: Loss = 2.2112


In [ ]:
logits = W[xdev] 
counts = logits.exp() 
probs = counts / counts.sum(1, keepdim=True) # 修正為 keepdim

# 這裡的 loss 同時包含了 NLL 和 L2 正則化
loss = -probs[torch.arange(dev_num), ydev].log().mean() + 0.01 * (W**2).mean()

print(f"Validation Data Loss = {loss.item():.4f}")

Validation Data Loss = 2.3977


In [104]:
logits = W[xtest] 
counts = logits.exp() 
probs = counts / counts.sum(1, keepdim=True) # 修正為 keepdim

# 這裡的 loss 同時包含了 NLL 和 L2 正則化
loss = -probs[torch.arange(test_num), ytest].log().mean() + 0.01 * (W**2).mean()

print(f"Test Data Loss = {loss.item():.4f}")

Test Data Loss = 2.2651


##### E04: Original: Use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?  
##### 中文翻譯： 使用驗證集（dev set）來微調 trigram 模型的平滑化（smoothing，針對計數模型）或正則化（regularization，針對神經網路）強度——也就是嘗試多種不同的超參數組合，並根據驗證集上的 Loss 找出效果最好的一個。當你在調整強度時，訓練集（train）與驗證集（dev）的 Loss 出現了什麼規律？選出最佳的平滑/正則化設定，並在最後對測試集（test set）進行一次性評估。你最終達到了多好的 Loss？

In [ ]:
# 想嘗試的正則化強度清單 (超參數列表)
lambdas = [0.0001, 0.001, 0.01, 0.1, 1.0]
best_dev_loss = float('inf')
best_lambda = None

for lam in lambdas:
    # 1. 重新初始化 W
    W = torch.randn((729, 27), generator=g, requires_grad=True)
    
    # 2. 【訓練階段】只在 Train Set 上做 gradient descent
    for step in range(2000):
        # Forward pass on TRAIN
        logits = W[xs] # (E04 優化寫法)
        loss = F.cross_entropy(logits, ys) + lam * (W**2).mean() # (E05 寫法)
        
        # Backward pass (只有訓練集需要！)
        W.grad = None
        loss.backward()
        W.data -= 50.0 * W.grad
        
    # 3. 【評估階段】在 Dev Set 上算 Loss (完全不改 W，也不做 backward)
    with torch.no_grad(): # 告訴 PyTorch 不用算梯度，省記憶體
        dev_logits = W[xdev]
        dev_loss = F.cross_entropy(dev_logits, ydev) # 評估時通常不加正則化項，只看真正的 NLL
        
    print(f"Lambda: {lam:<6} | Dev Loss: {dev_loss.item():.4f}")
    
    # 記下表現最好的超參數
    if dev_loss.item() < best_dev_loss:
        best_dev_loss = dev_loss.item()
        best_lambda = lam

print(f"\n最佳的超參數為 Lambda = {best_lambda}，Dev Loss 為 {best_dev_loss:.4f}")

with torch.no_grad(): # 告訴 PyTorch 不用算梯度，省記憶體
    test_logits = W[xtest]
    test_loss = F.cross_entropy(dev_logits, ytest) # 評估時通常不加正則化項，只看真正的 NLL

print('test_loss')

Lambda: 0.0001 | Dev Loss: 2.2362
Lambda: 0.001  | Dev Loss: 2.2364
Lambda: 0.01   | Dev Loss: 2.2379
Lambda: 0.1    | Dev Loss: 2.2527
Lambda: 1.0    | Dev Loss: 2.3996

最佳的超參數為 Lambda = 0.0001，Dev Loss 為 2.2362




##### E05: Original: Look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?
##### 中文翻譯： 查閱並改用 PyTorch 內建的 F.cross_entropy 函式。你應該會得到完全相同的結果。你能思考一下為什麼我們在實務上會更偏好使用 F.cross_entropy 嗎？